# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iqralodhi7866/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [21]:
%pip -q install duckdb huggingface_hub


In [22]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [23]:
import pandas as pd

# Target column check aur creation ke liye safe code:
if 'needs_refresh' in df.columns:
    print(df[['content_id', 'needs_refresh']].head())
else:
    # Check karein ki traffic drop ke liye kaun sa column available hai ('trend_pct' ya koi aur)
    target_col = 'trend_pct' if 'trend_pct' in df.columns else df.columns[-1]

    # Target column banayein
    df['needs_refresh'] = (df[target_col] < -25.0).astype(int)
    print(f"Target column successfully sketched using '{target_col}':")
    display(df[['content_id', target_col, 'needs_refresh']].head())

             content_id  needs_refresh
0  content_304f48230142              1
1  content_a1fb4e703a9e              1
2  content_9aa793d4d895              1
3  content_331d6c4de07b              0
4  content_d99b7a2d90ca              1


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [30]:
import duckdb
import os

# Ensure HF_TOKEN is loaded from Colab secrets or environment
if 'HF_TOKEN' not in os.environ:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# DuckDB mein Hugging Face secret configure karne ka correct syntax:
con.sql(f"CREATE OR REPLACE SECRET hf_secret (TYPE 'huggingface', TOKEN '{os.environ['HF_TOKEN']}');")
print("Hugging Face secret successfully configured in DuckDB!")

Hugging Face secret successfully configured in DuckDB!


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [34]:
import duckdb
import os

# 1. Token load karein
if 'HF_TOKEN' not in os.environ:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# 2. Fresh connection banayein
con = duckdb.connect()

# 3. Secret register karein
con.sql(f"CREATE OR REPLACE SECRET hf_secret (TYPE 'huggingface', TOKEN '{os.environ['HF_TOKEN']}');")

# 4. Tables map load karein (agar ye define nahi hain toh)
TABLES = {
    'fact_daily_sample': "read_parquet('hf://datasets/FlyRank/internship-warehouse@main/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse@main/fact_content_query_90d.parquet')"
}

print("DuckDB connected and authenticated successfully!")

DuckDB connected and authenticated successfully!


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [41]:
import os
import duckdb

# 1. Ensure HF_TOKEN is loaded and applied to DuckDB
if 'HF_TOKEN' not in os.environ:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

con.sql(f"CREATE OR REPLACE SECRET hf_secret (TYPE 'huggingface', TOKEN '{os.environ['HF_TOKEN']}');")

# 2. Table mappings define karein (agar pehle define nahi thin)
TABLES = {
    'fact_daily_sample': "read_parquet('hf://datasets/FlyRank/internship-warehouse@main/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse@main/fact_content_query_90d.parquet')"
}

# 3. Features dataframe generate karein
print("Building features dataframe...")
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily_sample']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily_sample']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()
print(f'{len(features):,} content items with enough history')

# 4. Query signals fetch karein aur merge karein
print("Fetching query signals and merging...")
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined successfully: {len(data):,} rows')
data.head()

Building features dataframe...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

0 content items with enough history
Fetching query signals and merging...
joined successfully: 0 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [44]:
import os
import duckdb
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. Ensure HF_TOKEN is loaded and applied to DuckDB
if 'HF_TOKEN' not in os.environ:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

con.sql(f"CREATE OR REPLACE SECRET hf_secret (TYPE 'huggingface', TOKEN '{os.environ['HF_TOKEN']}');")

# 2. Table mappings define karein
TABLES = {
    'fact_daily_sample': "read_parquet('hf://datasets/FlyRank/internship-warehouse@main/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse@main/fact_content_query_90d.parquet')"
}

# 3. Features dataframe generate karein (Flexible date windowing & threshold >= 0)
print("Building features dataframe...")
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily_sample']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily_sample']} f, bounds b
        GROUP BY 1, 2
        HAVING imp_prev30 >= 0
    )
    SELECT * FROM windowed
""").df()
print(f'{len(features):,} content items with enough history')

# 4. Query signals fetch karein aur merge karein
print("Fetching query signals and merging...")
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined successfully: {len(data):,} rows')

# 5. Train Model & Evaluate (Fallback handle kiya hai agar target class single ho)
data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

if len(y.unique()) > 1 and len(model_data) > 10:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

    print(f'\nBase rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
    print(classification_report(y_te, model.predict(X_te), digits=3))
else:
    print(f"\nModel data shape: {model_data.shape}. Unique classes in y: {y.unique()}")
    print("Please check data distribution or try running with full dataset if sample size is too small.")

Building features dataframe...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

409,205 content items with enough history
Fetching query signals and merging...
joined successfully: 409,205 rows

Model data shape: (133852, 13). Unique classes in y: [0]
Please check data distribution or try running with full dataset if sample size is too small.


Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.
